# Airbnb Paris — Prédiction du prix par nuitée

**Notebook principal à rendre** (Personne A + B)

## Plan
1. Chargement & nettoyage des données
2. Feature engineering
3. Pipeline (ColumnTransformer)
4. Modélisation & comparaison
5. Évaluation (CV 5-fold)
6. Analyse des résidus
7. Export du modèle final

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_validate, train_test_split

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Chargement & Nettoyage

In [ ]:
df = pd.read_csv("../data/listings.csv", low_memory=False)
print(f"Shape brut: {df.shape}")
df.head()

In [ ]:
# Parsing du prix : "$1,234.00" → float
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Filtrer les prix aberrants
df = df[(df["price"] > 0) & (df["price"] < 10_000)]
print(f"Shape après filtrage: {df.shape}")

## 2. Feature Engineering

In [ ]:
import ast

# Parsing amenities
df["amenities_list"] = df["amenities"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Créer au moins 5 features booléennes à partir des amenities
selected_amenities = ["Wifi", "TV", "Elevator", "Washer", "Dryer", "Air conditioning", "Balcony"]

for amenity in selected_amenities:
    col = "has_" + amenity.lower().replace(" ", "_").replace("/", "_")
    df[col] = df["amenities_list"].apply(lambda x, a=amenity: int(a in x))

In [ ]:
# Sélection des features
numeric_features = ["accommodates", "bedrooms", "beds", "bathrooms"]
categorical_features = ["neighbourhood_cleansed", "room_type", "property_type"]
amenity_features = [c for c in df.columns if c.startswith("has_")]

all_features = numeric_features + categorical_features + amenity_features

# Nettoyage NaN
df[numeric_features] = df[numeric_features].fillna(df[numeric_features].median())
df[categorical_features] = df[categorical_features].fillna("Unknown")

X = df[all_features]
y = df["price"]

print(f"X: {X.shape}, y: {y.shape}")

## 3. Pipeline (ColumnTransformer)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features + amenity_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)

## 4. Modélisation & Comparaison (CV 5-fold)

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    
    # TransformedTargetRegressor avec log1p/expm1
    ttr = TransformedTargetRegressor(
        regressor=pipe, func=np.log1p, inverse_func=np.expm1
    )
    
    scoring = {"r2": "r2", "mae": "neg_mean_absolute_error"}
    cv_results = cross_validate(ttr, X, y, cv=5, scoring=scoring)
    
    r2_mean = cv_results["test_r2"].mean()
    r2_std = cv_results["test_r2"].std()
    mae_mean = -cv_results["test_mae"].mean()
    mae_std = cv_results["test_mae"].std()
    
    results[name] = {"r2_mean": r2_mean, "r2_std": r2_std, "mae_mean": mae_mean, "mae_std": mae_std}
    print(f"{name:20s} | R² = {r2_mean:.4f} ± {r2_std:.4f} | MAE = {mae_mean:.2f}€ ± {mae_std:.2f}€")

In [ ]:
# Tableau récap
results_df = pd.DataFrame(results).T
results_df.sort_values("r2_mean", ascending=False)

## 5. Analyse des résidus (meilleur modèle)

In [ ]:
# Entraîner le meilleur modèle sur train/test split
best_name = results_df["r2_mean"].idxmax()
print(f"Meilleur modèle: {best_name}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

best_pipe = Pipeline([("preprocessor", preprocessor), ("model", models[best_name])])
best_ttr = TransformedTargetRegressor(regressor=best_pipe, func=np.log1p, inverse_func=np.expm1)
best_ttr.fit(X_train, y_train)

y_pred = best_ttr.predict(X_test)
residuals = y_test - y_pred

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred, residuals, alpha=0.3, s=5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Prédiction (€)")
axes[0].set_ylabel("Résidu (€)")
axes[0].set_title("Résidus vs Prédictions")

axes[1].hist(residuals, bins=80)
axes[1].set_title("Distribution des résidus")

axes[2].scatter(y_test, y_pred, alpha=0.3, s=5)
axes[2].plot([0, y_test.max()], [0, y_test.max()], "r--")
axes[2].set_xlabel("Prix réel (€)")
axes[2].set_ylabel("Prix prédit (€)")
axes[2].set_title("Réel vs Prédit")

plt.tight_layout()
plt.show()

## 6. Export du modèle & métriques

In [ ]:
# Ré-entraîner sur tout le dataset
final_pipe = Pipeline([("preprocessor", preprocessor), ("model", models[best_name])])
final_ttr = TransformedTargetRegressor(regressor=final_pipe, func=np.log1p, inverse_func=np.expm1)
final_ttr.fit(X, y)

# Sauvegarder
joblib.dump(final_ttr, "../model.joblib")
print("Modèle sauvegardé → model.joblib")

# Métriques
metrics = {
    "best_model": best_name,
    "cv_r2_mean": results[best_name]["r2_mean"],
    "cv_r2_std": results[best_name]["r2_std"],
    "cv_mae_mean": results[best_name]["mae_mean"],
    "cv_mae_std": results[best_name]["mae_std"],
}

with open("../metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Métriques sauvegardées → metrics.json")
metrics